# 07 — Mortality Outcomes (RQ5)

**Research Question:** Can we reduce mortality?

Longer stays correlate with higher mortality and complications. We quantify the
LOS-mortality relationship and estimate lives saveable through efficiency improvements.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from shared import (
    load_kidney, load_hospital_tags, setup_plot_style,
    save_plot, save_metrics,
)

setup_plot_style()
kidney = load_kidney()
tags = load_hospital_tags()
print(f"Loaded {len(kidney):,} admissions, {len(tags)} hospitals")

total_deaths = kidney["MORTE"].sum()
mortality_rate = kidney["MORTE"].mean() * 100
print(f"Total deaths: {total_deaths:,}")
print(f"Overall mortality rate: {mortality_rate:.3f}%")

Loaded 206,500 admissions, 510 hospitals
Total deaths: 714
Overall mortality rate: 0.346%


## 1. Mortality Trend Over Time

In [2]:
yearly_mort = kidney.groupby("year").agg(
    deaths=pd.NamedAgg(column="MORTE", aggfunc="sum"),
    mortality_rate=pd.NamedAgg(column="MORTE", aggfunc="mean"),
    admissions=pd.NamedAgg(column="year", aggfunc="count"),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(yearly_mort["year"], yearly_mort["deaths"], color="#DC2626")
axes[0].set_title("Deaths per Year")
axes[0].set_ylabel("Count")

axes[1].plot(yearly_mort["year"], yearly_mort["mortality_rate"] * 100, "o-", color="#DC2626")
axes[1].set_title("Mortality Rate Trend")
axes[1].set_ylabel("%")

for ax in axes:
    ax.set_xlabel("Year")
    ax.tick_params(axis="x", rotation=45)

fig.suptitle("Kidney Stone Mortality Over Time", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "mortality_trend", prefix="07")

  Saved: 07_mortality_trend.png


## 2. LOS-Mortality Gradient

How does mortality increase with length of stay?

In [3]:
# LOS bins
los_bins = [0, 1, 2, 3, 4, 5, 7, 10, 14, 21, 100]
los_labels = ["0-1", "1-2", "2-3", "3-4", "4-5", "5-7", "7-10", "10-14", "14-21", "21+"]
kidney["los_bin"] = pd.cut(kidney["DIAS_PERM"], bins=los_bins, labels=los_labels, right=False)

los_mort = kidney.groupby("los_bin", observed=False).agg(
    mortality_rate=pd.NamedAgg(column="MORTE", aggfunc="mean"),
    deaths=pd.NamedAgg(column="MORTE", aggfunc="sum"),
    count=pd.NamedAgg(column="MORTE", aggfunc="count"),
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(len(los_mort)), los_mort["mortality_rate"] * 100, color="#DC2626")
axes[0].set_xticks(range(len(los_mort)))
axes[0].set_xticklabels(los_labels, rotation=45)
axes[0].set_title("Mortality Rate by LOS")
axes[0].set_xlabel("Length of Stay (days)")
axes[0].set_ylabel("Mortality Rate (%)")

axes[1].bar(range(len(los_mort)), los_mort["deaths"], color="#D97706")
axes[1].set_xticks(range(len(los_mort)))
axes[1].set_xticklabels(los_labels, rotation=45)
axes[1].set_title("Absolute Deaths by LOS")
axes[1].set_xlabel("Length of Stay (days)")
axes[1].set_ylabel("Deaths")

fig.suptitle("LOS-Mortality Gradient", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "los_mortality_gradient", prefix="07")

print(los_mort.to_string())

  Saved: 07_los_mortality_gradient.png
         mortality_rate  deaths  count
los_bin                               
0-1            0.000884      23  26023
1-2            0.000948      60  63282
2-3            0.001328      71  53468
3-4            0.002365      61  25797
4-5            0.003335      45  13493
5-7            0.006054      72  11892
7-10           0.013244      93   7022
10-14          0.028352      85   2998
14-21          0.062616     101   1613
21+            0.112939     103    912


## 3. Mortality by Procedure Type

In [4]:
proc_mort = kidney.groupby("proc_category").agg(
    mortality_rate=pd.NamedAgg(column="MORTE", aggfunc="mean"),
    deaths=pd.NamedAgg(column="MORTE", aggfunc="sum"),
    count=pd.NamedAgg(column="MORTE", aggfunc="count"),
    mean_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="mean"),
).sort_values("mortality_rate", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
(proc_mort["mortality_rate"] * 100).plot.barh(ax=ax, color="#DC2626")
ax.set_title("Mortality Rate by Procedure Category")
ax.set_xlabel("Mortality Rate (%)")
ax.invert_yaxis()
fig.tight_layout()
save_plot(fig, "mortality_by_procedure", prefix="07")
print(proc_mort.to_string())

  Saved: 07_mortality_by_procedure.png
                mortality_rate  deaths  count  mean_los
proc_category                                          
OTHER                 0.017852      63   3529  4.030037
SURGICAL              0.004601     408  88681  2.614777
CLINICAL_MGMT         0.003308      77  23275  2.388786
SURGICAL_MGMT         0.002088      43  20597  2.247852
DIAGNOSTIC            0.001904      79  41487  2.687734
INTERVENTIONAL        0.001740      35  20113  2.129916
OBSERVATION           0.001021       9   8818  0.580517


## 4. Age-Stratified Risk

In [5]:
age_bins = [0, 30, 40, 50, 60, 70, 80, 100]
age_labels = ["<30", "30-39", "40-49", "50-59", "60-69", "70-79", "80+"]
kidney["age_group"] = pd.cut(kidney["age"], bins=age_bins, labels=age_labels, right=False)

age_mort = kidney.groupby("age_group", observed=False).agg(
    mortality_rate=pd.NamedAgg(column="MORTE", aggfunc="mean"),
    deaths=pd.NamedAgg(column="MORTE", aggfunc="sum"),
    count=pd.NamedAgg(column="MORTE", aggfunc="count"),
    mean_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="mean"),
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(age_labels, age_mort["mortality_rate"] * 100, color="#DC2626")
axes[0].set_title("Mortality Rate by Age Group")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Mortality Rate (%)")

axes[1].bar(age_labels, age_mort["mean_los"], color="#D97706")
axes[1].set_title("Mean LOS by Age Group")
axes[1].set_xlabel("Age")
axes[1].set_ylabel("Days")

fig.suptitle("Age-Stratified Risk", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "age_stratified_risk", prefix="07")
print(age_mort.to_string())

  Saved: 07_age_stratified_risk.png
           mortality_rate  deaths  count  mean_los
age_group                                         
<30              0.000486      15  30843  2.448789
30-39            0.001014      40  39460  2.324075
40-49            0.001505      72  47826  2.330511
50-59            0.002876     130  45201  2.415367
60-69            0.006484     195  30074  2.550276
70-79            0.014503     157  10825  3.105681
80+              0.046235     105   2271  4.084985


## 5. ICU Utilization

In [6]:
if "MARCA_UTI" in kidney.columns:
    kidney["used_icu"] = kidney["MARCA_UTI"].astype(str).str.strip() != "00"
    icu_stats = kidney.groupby("used_icu").agg(
        count=pd.NamedAgg(column="MORTE", aggfunc="count"),
        mortality_rate=pd.NamedAgg(column="MORTE", aggfunc="mean"),
        mean_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="mean"),
    )
    print("ICU usage statistics:")
    print(icu_stats.to_string())

    icu_pct = kidney["used_icu"].mean() * 100
    print(f"\n{icu_pct:.1f}% of admissions used ICU")
else:
    print("MARCA_UTI column not available.")
    icu_pct = 0

ICU usage statistics:
           count  mortality_rate  mean_los
used_icu                                  
False     201422        0.001475  2.304078
True        5078        0.082119  8.541355

2.5% of admissions used ICU


## 6. Estimate: Lives Saveable

If LOS for long-stay patients (>7d) were reduced to 7 days, how many deaths would be avoided?
This is a **crude estimate** based on the LOS-mortality gradient — not a causal claim.

In [7]:
# Mortality rate for stays <=7 days vs >7 days
short_stay_mort = kidney[kidney["DIAS_PERM"] <= 7]["MORTE"].mean()
long_stay_mort = kidney[kidney["DIAS_PERM"] > 7]["MORTE"].mean()
n_long = len(kidney[kidney["DIAS_PERM"] > 7])
deaths_long = kidney[kidney["DIAS_PERM"] > 7]["MORTE"].sum()

# If long-stay patients had short-stay mortality rate
expected_deaths_if_short = n_long * short_stay_mort
lives_saveable = deaths_long - expected_deaths_if_short

print(f"Mortality rate (<=7d): {short_stay_mort*100:.3f}%")
print(f"Mortality rate (>7d): {long_stay_mort*100:.3f}%")
print(f"Long-stay patients: {n_long:,}")
print(f"Deaths in long-stay: {deaths_long:,}")
print(f"Expected deaths at short-stay rate: {expected_deaths_if_short:.0f}")
print(f"Estimated lives saveable: {lives_saveable:.0f}")
print(f"\nCAUTION: This is correlation, not causation. Sicker patients stay longer.")

Mortality rate (<=7d): 0.181%
Mortality rate (>7d): 3.826%
Long-stay patients: 9,330
Deaths in long-stay: 357
Expected deaths at short-stay rate: 17
Estimated lives saveable: 340

CAUTION: This is correlation, not causation. Sicker patients stay longer.


## Summary Metrics

In [8]:
metrics = {
    "total_deaths": int(total_deaths),
    "overall_mortality_rate": round(float(mortality_rate), 3),
    "short_stay_mortality_rate": round(float(short_stay_mort * 100), 3),
    "long_stay_mortality_rate": round(float(long_stay_mort * 100), 3),
    "mortality_ratio_long_vs_short": round(float(long_stay_mort / max(short_stay_mort, 1e-6)), 1),
    "lives_saveable_estimate": int(round(lives_saveable)),
    "icu_utilization_pct": round(float(icu_pct), 1),
}
save_metrics(metrics, "07_mortality_outcomes")

print("\n=== MORTALITY OUTCOMES ===")
for k, v in metrics.items():
    print(f"  {k}: {v}")

  Saved metrics: 07_mortality_outcomes.json

=== MORTALITY OUTCOMES ===
  total_deaths: 714
  overall_mortality_rate: 0.346
  short_stay_mortality_rate: 0.181
  long_stay_mortality_rate: 3.826
  mortality_ratio_long_vs_short: 21.1
  lives_saveable_estimate: 340
  icu_utilization_pct: 2.5
